In [60]:
from pathlib import Path
from enum import Enum
import hashlib
import csv
import json
import docx
import pypdf

class FileType(Enum):
    DOCX = ".docx"
    PDF = ".pdf"
    CSV = ".csv"
    JSON = ".json"
    TXT = ".txt"
    MD = ".md"

def string_to_hash_id(source_file: str, source_type: str, block_id: int, text: str) -> str:
    key = f"{source_file}::{source_type}::{block_id}::{text}"
    return hashlib.sha256(key.encode("utf-8")).hexdigest()

def parse_document(file_path: str | Path) -> str:
    path = Path(file_path)
    ext = path.suffix.lower()
    file_str = str(path)
    blocks: list[str] = []

    try:
        file_type = FileType(ext)
    except ValueError:
        raise ValueError(f"Unsupported format: {ext}")

    match file_type:
        # 1. DOCX
        case FileType.DOCX:
            doc = docx.Document(path)
            text = "\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])
            if text:
                blocks.append(text)

        # 2. PDF
        case FileType.PDF:
            reader = pypdf.PdfReader(path)
            page_texts = [
                page.extract_text().strip()
                for page in reader.pages
                if page.extract_text() and page.extract_text().strip()
            ]
            full_text = "\n\n".join(page_texts)
            if full_text:
                blocks.append(full_text)

        # 3. CSV
        case FileType.CSV:
            with open(path, mode="r", encoding="utf-8", errors="ignore") as f:
                reader = csv.reader(f)
                header = next(reader, None)

                if header:
                    for row in reader:
                        if row:
                            paired_row = [
                                f"{h.strip()}: {v.strip()}"
                                for h, v in zip(header, row)
                            ]
                            blocks.append(" | ".join(paired_row))

        # 4. JSON
        case FileType.JSON:
            with open(path, mode="r", encoding="utf-8", errors="ignore") as f:
                data = json.load(f)
                if isinstance(data, list):
                    blocks = [json.dumps(item, ensure_ascii=False) for item in data]
                else:
                    blocks = [json.dumps(data, indent=2, ensure_ascii=False)]

        # 5. TXT / MD
        case FileType.TXT | FileType.MD:
            with open(path, mode="r", encoding="utf-8", errors="ignore") as f:
                content = f.read().strip()
                if content:
                    blocks.append(content)

    output_blocks = []
    for i, block in enumerate(blocks):
        output_blocks.append(
            {
                "id": string_to_hash_id(file_str, ext, i, block),
                "source_file": file_str,
                "source_type": ext,
                "text": block,
            }
        )

    return output_blocks

In [61]:
from RAG.Pipeline.Error_Handler import call_safe_function
from google import genai
from dotenv import load_dotenv
import numpy as np

load_dotenv()
client = genai.Client()

# Normalized vector helps with cosine similarity.
def normalize_embedding(embedding):
    print("Normalizing Semantic Embeddings...")
    vectors = np.array([e.values for e in embedding.embeddings])
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

def embed(texts: list[str], model = "gemini-embedding-001", config=None):
    print(f"[embed] {len(texts)} texts, 1 request")
    return call_safe_function(
        client.models.embed_content,
        model = model,
        contents = texts,
        config = config or {"output_dimensionality": 768},
    )

def generate_response(prompt: str, model="gemini-3-flash", config=None):
    """
    config example: {
        "system_instruction": "You are a helpful assistant.",
        "temperature": 0.7,
        "max_output_tokens": 1024,
    }
    """
    return call_safe_function(
        client.models.generate_content,
        model = model,
        contents = prompt,
        config = config,
    )

In [62]:
parsed_blocks = parse_document('../Sample Files/sample2.docx')
for block in parsed_blocks:
    print(f"{block["text"]}\n")

PROFILE
AI Engineering student at Alexandria University's Faculty of Computer and Data Science, with a strong dual specialisation in machine learning systems and 3D visual technology. Experienced in building neural networks from first principles, exploring quantum computing, and creating production-quality 3D assets and game systems. Comfortable across the full pipeline — from training classifiers and designing robotic locomotion to hard-surface modelling, procedural generation, and real-time game mechanics.
EDUCATION
B.Sc. Artificial Intelligence	Expected June 2027
Alexandria University — Faculty of Computer and Data Science (FCDS)
Relevant coursework: Neural Networks, Deep Learning, Quantum Computing, Web Development, Robotics & Control Systems
SKILLS
AI & Programming
Languages:  Python   ·   Lua   ·   Java   ·   C#   ·   C++
AI / ML:  Neural Networks from Scratch   ·   Deep Learning   ·   Classification   ·   Data Preprocessing   ·   Model Evaluation
Quantum:  Qiskit (Bronze · Silve

In [63]:
import re
import tiktoken
from sklearn.feature_extraction.text import TfidfVectorizer
from RAG.Pipeline.Gemini_client import embed, normalize_embedding

encoder = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(encoder.encode(text))

def _split_sentences(text: str) -> list[str]:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

class ChunkMethod(Enum):
    NAIVE = "naive"
    STATISTICAL = "statistical"
    SEMANTIC = "semantic"

# Takes input text and vectors (whatever they may be) and performs cosine similary on them and then chunks them.
def _chunkify(sentences: list[str], vectors: np.ndarray, threshold: float,
              Max_Tokens: int = 400, Min_Tokens: int = 100) -> list[str]:
    chunks = []
    current_chunk = [sentences[0]]
    cumulative_token_count = count_tokens(sentences[0])

    for i in range(1, len(sentences)):
        a, b = vectors[i - 1], vectors[i]
        sim = np.dot(a, b)
        sentence_token_count = count_tokens(sentences[i])

        should_break = sim < threshold or cumulative_token_count + sentence_token_count > Max_Tokens
        below_min = cumulative_token_count < Min_Tokens

        if should_break and not below_min:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
            cumulative_token_count = sentence_token_count
        else:
            current_chunk.append(sentences[i])
            cumulative_token_count += sentence_token_count

    chunks.append(" ".join(current_chunk))
    return chunks

def chunk_naive(text: str, chunk_size: int, overlap: int) -> list[str]:
    words  = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start: end]))
        start += chunk_size - overlap
    return chunks

def chunk_statistic(text: str, threshold: float = 0.6, min_tokens: int = 100, max_tokens: int = 400) -> list[str]:
    sentences = _split_sentences(text)
    if len(sentences) <= 1:
        return [text.strip()] if text.strip() else []
    
    vectors = TfidfVectorizer().fit_transform(sentences).toarray()
    return _chunkify(sentences, vectors, threshold, Max_Tokens = max_tokens, Min_Tokens = min_tokens)

def chunk_semantic(text: str, threshold: float = 0.6, min_tokens: int = 100, max_tokens: int = 400) -> list[str]:
    sentences = _split_sentences(text)
    if len(sentences) <= 1:
        return [text.strip()] if text.strip() else []

    response = embed(sentences)
    vectors = normalize_embedding(response)
    return _chunkify(sentences, vectors, threshold, Max_Tokens=max_tokens, Min_Tokens=min_tokens)

def chunk_block(block: dict, strategy: ChunkMethod) -> list[str]:
    if block["source_type"] == ".csv":
        return [block["text"]]

    match strategy:
        case ChunkMethod.NAIVE:
            return chunk_naive(block["text"], chunk_size=50, overlap=5)
        case ChunkMethod.STATISTICAL:
            return chunk_statistic(block["text"])
        case ChunkMethod.SEMANTIC:
            return chunk_semantic(block["text"])
        case _:
            raise ValueError(f"Unknown strategy: {strategy}")

In [64]:
# chunks = []
# for block in parsed_blocks:
#     chunks += chunk_statistic(block["text"], 50, 5)
    
# for chunk in chunks:
#     print("=>", chunk, "\n")

In [65]:
def build_chunks(block: dict, method: ChunkMethod) -> list[dict]:
    """Chunk a parsed block's text and return chunk dicts matching parse_document's structure."""
    text_chunks = chunk_block(block, method)

    output_chunks = []
    for i, chunk_text in enumerate(text_chunks):
        output_chunks.append(
            {
                "id": string_to_hash_id(block["source_file"], block["source_type"], i, chunk_text),
                "source_file": block["source_file"],
                "source_type": block["source_type"],
                "text": chunk_text,
                "vector": None,
            }
        )
    return output_chunks

In [ ]:
from RAG.Pipeline.VectorDB import store_chunks, query, drop
from fastapi import FastAPI

#uvicorn RAG.Pipeline.Orchestrator:api --reload
api = FastAPI()

@api.post('/ingest')
def ingest_file(path: str, method: str = "semantic"):
    parsed_blocks = parse_document(path)
    method = ChunkMethod(method)

    data_chunks = []
    for block in parsed_blocks:
        data_chunks += build_chunks(block, method)

    texts = [c["text"] for c in data_chunks]
    embedding_result = embed(texts)

    for chunk, embedding in zip(data_chunks, embedding_result.embeddings):
        chunk["vector"] = embedding.values

    store_chunks(data_chunks)
    return {"chunks ingested": len(data_chunks)}

@api.get('/query')
def ask(question: str, top_k: int = 3):
    query_vector = embed([question]).embeddings[0].values
    results = query(query_vector, top_k)
    return results

@api.post('/drop')
def drop_table(table_name: str = "rag_chunks", persist_path: str = "./RAG/lancedb_store"):
    drop(table_name, persist_path)

#api.get('/generate)
def generate_query_reply(question: str, top_k: int = 3):
    query_vector = embed([question]).embeddings[0].values
    results = query(query_vector, top_k)
    context = build_prompt_context(results)

    #gemini-3-flash
    #gemini-3.1-flash-lite

    response = generate_response(
        prompt=f"Context:\n{context}\nQuestion: {question}",
        model = "gemini-3.1-flash-lite",
        config={
            "system_instruction": (
                "You are a helpful assistant. Answer using only the provided context. "
                "If the answer isn't in the context, say you don't know."
            ),
            "temperature": 0.3,
        },
    )
    return response.text

def build_prompt_context(results: list[dict]) -> str:
    return "\n".join(
    f"Source:{r["source_file"]}\n{r["text"]}\n{r["metadata"]}"
    for r in results
    )

In [67]:
print("Dropping old Data..")
drop_table()

#statistic / semantic
print("Starting Ingestion Process...")
ingest_file("../Sample Files/sample2.docx", method = "semantic")
print("Ingestion Complete!")

#question = "What software does sohayl use?"
#question = "Where is sohayls profile?"
# question = "what are sohayl's projects?"

# print(f"QueryDB results:\n")
# qr = ask(question, top_k = 5)
# for r in qr:
# #     #print(r["id"], r["text"], r["metadata"], r["_distance"])
#     print("-",r["text"], r["_distance"])

# print("QueryAPI Request pending...")
# result = generate_query_reply(question, top_k = 5)

# print(f"Query:{question} \nResults:{result}")



Dropping old Data..
Starting Ingestion Process...
[embed] 26 texts, 1 request
Normalizing Semantic Embeddings...
[embed] 6 texts, 1 request
Ingestion Complete!
